In [2]:
import torch
from torch.amp import autocast
import time
import matplotlib.pyplot as plt
from v1_model import ContinuousMotionModel, load_model
from dataset.dataset import *
import utils.utils as utils
import soundfile as sf
from IPython.display import clear_output

device = utils.get_device()

In [19]:
# Get the newest model from the directory. That means find the newest folder, and then the newest file in that folder.
model_path = utils.get_latest_model_path("v1_models")
# model_path = "v1_models/first_tests_2025-05-30_17-28-27/first_tests_2025-05-30_17-28-27_epoch_100.pth"
print(f"Model path: {model_path}")

# Load the model
model: ContinuousMotionModel = load_model(model_path,device)
model.condition_mask_probabilty = 0.0  # Disable condition mask probability for inference
model = model.to(device)

Model path: v1_models\first_tests_2025-05-30_17-28-27\first_tests_2025-05-30_17-28-27_epoch_716.pth


In [ ]:
import utils.animation.visualisation.new.animation_visualisation as animation_visualisation
animation_visualisation.init_visualization()

Viewer URL: http://localhost:8001/utils/animation/visualisation/bvh_viewer.html?wsport=8701


In [ ]:
dataset = GPUDataset(
    consolidated_file="dataset/genea2023_dataset/val/main-agent/consolidated.npz",
    seq_length=100,
    seed_length=0,
    batch_size=1,
    epoch_length=1,  # Set to 1 for testing purposes
    return_audio_frame_index=True,  # Set to True to return the audio frame index
)

# Now I want to test the autoregressive generation of the model
# model = torch.compile(model, backend="cudagraphs")

# Use no gradient calculation for inference
with torch.no_grad():
    # Get a sample from the dataset
    # gesture_sequence, seed_gesture, audio_features, main_agent_id_one_hot, full_audio_features, start_frame, file = next(iter(dataset))
    gesture_sequence, seed_gesture, audio_features, main_agent_id_one_hot, start_frames = next(iter(dataset))
    gesture_sequence = gesture_sequence.to(device)
    seed_gesture = seed_gesture.to(device)
    audio_features = audio_features.to(device)
    main_agent_id_one_hot = main_agent_id_one_hot.to(device)
    full_audio_features = dataset.audio.to(device)
    start_frame = start_frames[0].item()  # Extract the first element from the tensor

    with autocast(device_type=device.type, dtype=torch.bfloat16):
        # Decode the output using the autoencoder model
        gesture_sequence_encoded = model.pose_encoder.encode(gesture_sequence)

    # Apply noise to the gesture_sequence using the diffusion process
    noisy_gesture_sequence = model.diffusion.forward(
        sequence_tensor=gesture_sequence_encoded,
        stacking_level=0,
    )

    iteration_counter = 0
    while True:
        # Start time for the current frame
        frame_start_time = time.time()

        # Shift the gesture_sequence by one frame
        shifted_gesture_sequence = torch.roll(noisy_gesture_sequence, shifts=-1, dims=1)
        
        # Replace the last frame with pure noise
        shifted_gesture_sequence[0,-1] = torch.zeros_like(shifted_gesture_sequence[0,-1])

        # Apply diffusion noise to the shifted_gesture_sequence
        shifted_gesture_sequence = model.diffusion.forward(sequence_tensor=shifted_gesture_sequence, stacking_level=0)

        # The audio features also have to be shifted by one frame
        # I have the full audio features and the starting frame, so I extract the audio features for the current frame
        actual_audio_features = full_audio_features[start_frame + iteration_counter: start_frame + iteration_counter + shifted_gesture_sequence.shape[1], :].unsqueeze(0)

        with autocast(device_type=device.type, dtype=torch.bfloat16):
            # Convert all inputs to same precision as autocast context

            # denoised_gesture_sequence = shifted_gesture_sequence
            denoised_gesture_sequence = model(
                time_step_stacking_level = 0,
                one_hot_style = main_agent_id_one_hot,
                audio_features = actual_audio_features, 
                noisy_gesture_sequence = shifted_gesture_sequence
            )
            if model.predict_full_duration:
                denoised_gesture_sequence[0, :model.num_of_pre_timestep_frames] = shifted_gesture_sequence[0, :model.num_of_pre_timestep_frames]
        
        # Clear old output
        clear_output(wait=True)

        with autocast(device_type=device.type, dtype=torch.bfloat16):
            # Decode the output using the autoencoder model
            denoised_gesture_sequence_unencoded = model.pose_encoder.decode(denoised_gesture_sequence)

        # Undo the z-score normalization
        unnormalized_gesture_sequence = dataset.skeleton.denormalize_poses(denoised_gesture_sequence_unencoded)

        animation_visualisation.send_pose(unnormalized_gesture_sequence[50].squeeze(0).cpu(), dataset.skeleton)
        animation_visualisation.send_debug_tensor(torch.cat((actual_audio_features.squeeze(0).to(torch.float32),denoised_gesture_sequence.squeeze(0).to(torch.float32)), dim=1), "full tensor")

        # Update the noisy_gesture_sequence for the next iteration
        noisy_gesture_sequence = denoised_gesture_sequence

        iteration_counter += 1

        print(f"Iteration: {iteration_counter}")
        frame_end_time = time.time()
        # Print the time taken for the current frame
        print(f"Frame {iteration_counter} processed in {frame_end_time - frame_start_time:.4f} seconds ({1/(frame_end_time - frame_start_time):.2f} FPS)")

        # Sleep for the remaining time in the 30 FPS frame
        time_to_sleep = max(0, (1/30) - (frame_end_time - frame_start_time) - 0.0005)  # 0.01 is a small buffer to account for processing time
        time.sleep(time_to_sleep)

KeyboardInterrupt: 